# 03 — Segmentation Training

Train **one fold** of one segmentation experiment, end-to-end. Everything about a run — model, loss, optimizer, scheduler, augmentation, fold — is in the `EXPERIMENT` dict in cell 3.

To switch experiments: edit cell 3, run all cells again. To train all 5 folds, change `EXPERIMENT["fold"]` and re-run from cell 3 (no kernel restart needed), or use the multi-fold loop at the bottom.

**Outputs per fold (after the final sync cell):**

```
senior_project/
├── configs/                            ← NEW
│   ├── __init__.py
│   └── seg/
│       ├── __init__.py
│       └── reference_experiments.py    ← the 7 §13 recipes
├── src/                                (unchanged)
└── notebooks/
    └── segmentation/
        └── 03_train.ipynb              ← cell 3 updated
```

**Runtime:** open on **GPU runtime** (T4 or better). One fold of `01_dice_image_level` typically takes 30–60 minutes.

## Cell 1 — Install dependencies

In [1]:
%pip install -q pytorch-lightning segmentation-models-pytorch albumentations opencv-python-headless timm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 48.3 MB/s eta 0:00:00


## Cell 2 — Bootstrap: Drive + repo + local SSD scratch

In [2]:
import os, sys

if not os.path.exists("/content/senior_project"):
    from google.colab import userdata
    try:
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        token = None
    url = "https://github.com/salemaker47/senior_project.git"
    if token:
        url = url.replace("https://", f"https://{token}@", 1)
    os.system(f"git clone {url} /content/senior_project")
if "/content/senior_project" not in sys.path:
    sys.path.insert(0, "/content/senior_project")

from src.notebook_setup import setup_environment

DRIVE_ROOT, REPO_ROOT = setup_environment(
    repo_url="https://github.com/salemaker47/senior_project.git",
)
# NOTE: copy_to_local moved to Cell 4 (after EXPERIMENT picks the dataset)
print(f"DRIVE_ROOT: {DRIVE_ROOT}")
print(f"REPO_ROOT:  {REPO_ROOT}")

Mounted at /content/drive
DRIVE_ROOT: /content/drive/MyDrive/Senior_Project
REPO_ROOT:  /content/senior_project


In [3]:
import os, psutil
print(f"CPU count:   {os.cpu_count()}")
print(f"RAM total:   {psutil.virtual_memory().total / 1e9:.1f} GB")
print(f"RAM avail:   {psutil.virtual_memory().available / 1e9:.1f} GB")

CPU count:   8
RAM total:   54.8 GB
RAM avail:   52.9 GB


## Cell 3 — EXPERIMENT dict (the single source of truth)

Edit this dict to define a run. Defaults match the FigShare reference reproduction's experiment 01.

**Common edits when iterating:**
- `fold`: 1 → 2 → 3 → 4 → 5 (run all 5)
- `name`: switch between the 7 reference experiments (see §13 of the project instruction)
- `loss_name`, `model_name`: swap registries
- `max_epochs`: drop to 3–5 for a quick smoke test before a real run

In [4]:
from configs.seg.reference_experiments import get_experiment, REFERENCE_RECIPES

# ----- The 4 axes of a run -----
RECIPE       = "06_clahe_dicebce"        # one of REFERENCE_RECIPES
DATASET      = "figshare"          # "figshare" | "brats2020"
SPLIT_SCHEME = "image_level"       # "image_level" | "patient_level"
# NOTE: fold is now controlled by Cell 5 (FOLD_TO_RUN / RUN_ALL_FOLDS),
# not here. The value below is just the registry default.

EXPERIMENT = get_experiment(
    RECIPE, dataset=DATASET, split_scheme=SPLIT_SCHEME, fold=1,
)

# ----- Dataset-specific throughput / training overrides ------------------------
# These keep §13 reproducibility intact for figshare (batch_size unchanged),
# and accept a tiny Dice penalty on brats2020 in exchange for ~2× speedup.
if EXPERIMENT["dataset"] == "figshare":
    EXPERIMENT["num_workers"] = 6                    # batch_size + lr untouched
elif EXPERIMENT["dataset"] == "brats2020":
    EXPERIMENT["batch_size"]             = 16        # was 8 (recipe default)
    EXPERIMENT["num_workers"]            = 6
    EXPERIMENT["optimizer_kwargs"]["lr"] = 1.4e-4    # sqrt(16/8) * 1e-4

# ----- Sanity asserts ---------------------------------------------------------
assert EXPERIMENT["task"] == "segmentation"
assert EXPERIMENT["dataset"] in ("figshare", "brats2020")
assert EXPERIMENT["split_scheme"] in ("image_level", "patient_level")

print(f"EXPERIMENT: {EXPERIMENT['name']}")
print(f"  recipe:       {EXPERIMENT['recipe']}")
print(f"  dataset:      {EXPERIMENT['dataset']}")
print(f"  split_scheme: {EXPERIMENT['split_scheme']}")
print(f"  model:        {EXPERIMENT['model_name']}")
print(f"  loss:         {EXPERIMENT['loss_name']}  kwargs={EXPERIMENT['loss_kwargs']}")
print(f"  prepro:       {EXPERIMENT['preprocessing']}  aug={EXPERIMENT['augmentation_strength']}")
print(f"  batch_size:   {EXPERIMENT['batch_size']}    num_workers: {EXPERIMENT['num_workers']}")
print(f"  lr:           {EXPERIMENT['optimizer_kwargs']['lr']}")
print(f"\nAvailable recipes: {REFERENCE_RECIPES}")

EXPERIMENT: 06_clahe_dicebce_image_level
  recipe:       06_clahe_dicebce
  dataset:      figshare
  split_scheme: image_level
  model:        smp_unet_resnet34
  loss:         dice_bce  kwargs={'bce_weight': 1.0, 'dice_weight': 1.0}
  prepro:       clahe  aug=reference
  batch_size:   8    num_workers: 6
  lr:           0.0001

Available recipes: ['01_dice', '02_bce', '03_dicebce', '04_dicefocal', '05_lovasz', '06_clahe_dicebce', '07_unetpp_effb4_dicebce']


In [5]:
import os, psutil
print(f"CPU count:   {os.cpu_count()}")
print(f"RAM total:   {psutil.virtual_memory().total / 1e9:.1f} GB")
print(f"RAM avail:   {psutil.virtual_memory().available / 1e9:.1f} GB")

CPU count:   8
RAM total:   54.8 GB
RAM avail:   52.8 GB


## Cell 4 — Seed + path resolution

In [6]:
from src.train_utils    import set_global_seed
from src.notebook_setup import copy_to_local
from src.file_utils     import experiment_paths, fold_split_csv_paths

set_global_seed(EXPERIMENT["seed"])
LOCAL_ROOT   = copy_to_local(DRIVE_ROOT, datasets=[EXPERIMENT["dataset"]])
PROJECT_ROOT = LOCAL_ROOT
print(f"LOCAL_ROOT: {LOCAL_ROOT}")


INFO:lightning_fabric.utilities.seed:Seed set to 42


[copy_to_local] copying /content/drive/MyDrive/Senior_Project/data/figshare -> /content/Senior_Project_local/data/figshare
[copy_to_local] cwd is now /content/Senior_Project_local
LOCAL_ROOT: /content/Senior_Project_local


## Cell 5 — Fold Control

In [7]:
# ----- Fold control --------------------------------------------------------
FOLD_TO_RUN   = 1         # used when RUN_ALL_FOLDS = False
RUN_ALL_FOLDS = True     # flip to True after fold 1 looks good
# ---------------------------------------------------------------------------

folds_to_run = list(range(1, 6)) if RUN_ALL_FOLDS else [FOLD_TO_RUN]
print(f"folds_to_run = {folds_to_run}")
print(f"(experiment: {EXPERIMENT['name']} on {EXPERIMENT['dataset']})")

folds_to_run = [1, 2, 3, 4, 5]
(experiment: 06_clahe_dicebce_image_level on figshare)


## Cell 6 — train_one_fold

In [8]:
import time
import torch
from pytorch_lightning.callbacks import ModelCheckpoint

from src.data_utils         import load_metadata, validate_metadata
from src.sg_data_utils      import build_dataloaders
from src.sg_models          import build_model, count_parameters
from src.sg_losses          import get_loss
from src.sg_lightning_module import BrainTumorSegModule
from src.train_utils        import (
    build_callbacks, build_trainer, TrainingTimingCallback,
    gather_repro_metadata, export_plain_state_dict,
)
from src.file_utils         import save_json


def train_one_fold(fold: int, experiment: dict) -> dict:
    """
    Train a single fold end-to-end.

    Reads fold CSVs from
        data/<dataset>/splits/<split_scheme>/folds/fold_<k>_{train,val}.csv

    Writes:
        outputs/checkpoints/<task>/<dataset>/<exp>/fold_<k>/best.ckpt
                                                          best_model.pt
                                                          experiment_config.json
        outputs/logs/<task>/<dataset>/<exp>/fold_<k>/lightning_logs/...
    """
    experiment["fold"] = fold

    # Resolve paths for this fold
    paths = experiment_paths(
        LOCAL_ROOT,
        task=experiment["task"], dataset=experiment["dataset"],
        experiment_name=experiment["name"], fold=fold,
    )
    csv_paths = fold_split_csv_paths(
        LOCAL_ROOT,
        dataset=experiment["dataset"],
        scheme=experiment["split_scheme"],
        fold=fold,
    )

    # ----- Data -----
    train_df = load_metadata(csv_paths["train"]); validate_metadata(train_df)
    val_df   = load_metadata(csv_paths["val"]);   validate_metadata(val_df)
    print(f"  fold {fold}: train={len(train_df):>5} imgs / "
          f"{train_df['patient_id'].nunique():>3} pts | "
          f"val={len(val_df):>5} imgs / {val_df['patient_id'].nunique():>3} pts")

    train_loader, val_loader = build_dataloaders(
        train_df=train_df, val_df=val_df, project_root=LOCAL_ROOT,
        batch_size=experiment["batch_size"],
        num_workers=experiment["num_workers"],
        image_size=experiment["image_size"],
        preprocessing=experiment["preprocessing"],
        augmentation_strength=experiment["augmentation_strength"],
        seed=experiment["seed"],
    )

    # ----- Model + loss + module -----
    model = build_model(
        experiment["model_name"],
        encoder_weights=experiment["encoder_weights"],
    )
    params_count = count_parameters(model)
    loss_fn = get_loss(experiment["loss_name"], **experiment.get("loss_kwargs", {}))

    pl_module = BrainTumorSegModule(
        model=model, loss_fn=loss_fn, threshold=experiment["threshold"],
        optimizer_name=experiment["optimizer_name"],
        optimizer_kwargs=experiment["optimizer_kwargs"],
        scheduler_name=experiment["scheduler_name"],
        scheduler_kwargs=experiment["scheduler_kwargs"],
        scheduler_monitor=experiment["scheduler_monitor"],
        metric_kind=experiment["metric_kind"],
    )

    # ----- Trainer + callbacks -----
    callbacks = build_callbacks(
        ckpt_dir=paths["checkpoints"], monitor="val_dice",
        mode="max", patience=experiment["patience"],
    )
    trainer = build_trainer(
        callbacks=callbacks, log_dir=paths["logs"],
        max_epochs=experiment["max_epochs"], precision="auto",
    )

    # ----- Train -----
    print(f"  fold {fold}: training start")
    t0 = time.time()
    trainer.fit(pl_module, train_loader, val_loader)
    train_seconds = time.time() - t0
    print(f"  fold {fold}: training done in {train_seconds/60:.1f} min")

    # ----- Collect training meta (Enhancement E) -----
    ckpt_cb   = next(c for c in callbacks if isinstance(c, ModelCheckpoint))
    timing_cb = next(c for c in callbacks if isinstance(c, TrainingTimingCallback))
    best_ckpt = ckpt_cb.best_model_path
    best_blob = torch.load(best_ckpt, map_location="cpu", weights_only=False)

    training_meta = {
        "fold":                 fold,
        "best_epoch":           int(best_blob.get("epoch", -1)),
        "best_val_dice":        float(ckpt_cb.best_model_score) if ckpt_cb.best_model_score is not None else None,
        "total_epochs_trained": int(trainer.current_epoch + 1),
        "train_seconds":        train_seconds,
        "params_count":         int(params_count),
        "peak_gpu_mem_mb":      timing_cb.peak_gpu_mem_mb,
        "best_ckpt_path":       best_ckpt,
    }

    # ----- Export plain .pt -----
    export_plain_state_dict(
        lightning_ckpt_path=best_ckpt,
        out_pt_path=paths["best_model"],
        extra_meta={
            "experiment":      experiment,
            "model_name":      experiment["model_name"],
            "encoder_weights": experiment["encoder_weights"],
        },
    )

    # ----- Save experiment_config.json (with Enhancements E + F) -----
    save_json(
        {
            "EXPERIMENT":     dict(experiment),
            "training_meta":  training_meta,
            "repro_metadata": gather_repro_metadata(repo_root=REPO_ROOT),
        },
        paths["experiment_config_json"],
    )

    # ----- Read history CSV for plotting later -----
    import pandas as pd
    metrics_csv = paths["logs"] / "lightning_logs" / "version_0" / "metrics.csv"
    history_df = pd.read_csv(metrics_csv) if metrics_csv.exists() else None

    return {
        **training_meta,
        "history":       history_df,
        "paths":         paths,
        "best_model_pt": str(paths["best_model"]),
    }

print("train_one_fold defined.")

train_one_fold defined.


## Cell 7 — The loop (NEW — with try/except/finally + gc/cuda)

In [ ]:
import gc

results = {}
for fold in folds_to_run:
    print(f"\n{'='*70}")
    print(f"  FOLD {fold}  ({EXPERIMENT['name']} on {EXPERIMENT['dataset']})")
    print(f"{'='*70}")
    try:
        results[fold] = train_one_fold(fold, EXPERIMENT)
        bvd = results[fold]['best_val_dice']
        print(f"\n  ✓ fold {fold} complete: best_val_dice = "
              f"{bvd:.4f}" if bvd is not None else f"  ✓ fold {fold} complete (no val_dice recorded)")
    except Exception as e:
        print(f"\n  ✗ fold {fold} FAILED: {type(e).__name__}: {e}")
        print(f"     to retry: set FOLD_TO_RUN={fold}, RUN_ALL_FOLDS=False in Cell 5, re-run from Cell 6")
        import traceback
        traceback.print_exc()
        results[fold] = {"error": str(e), "fold": fold}
    finally:
        # Force GC + CUDA cleanup between folds — combats memory fragmentation
        # and dataloader thread cleanup. Crucial before fold N+1's model loads.
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.synchronize()

print(f"\n{'='*70}")
print(f"  RUN COMPLETE — folds attempted: {list(results.keys())}")
print(f"  failed folds: {[k for k, v in results.items() if 'error' in v]}")
print(f"{'='*70}")


  FOLD 1  (06_clahe_dicebce_image_level on figshare)
  fold 1: train= 2083 imgs / 230 pts | val=  368 imgs / 167 pts


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/87.3M [00:00<?, ?B/s]

INFO:pytorch_lightning.utilities.rank_zero:Using 16bit Automatic Mixed Precision (AMP)
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


...

INFO:pytorch_lightning.callbacks.early_stopping:Metric val_dice improved by 0.002 >= min_delta = 0.0. New best score: 0.843


[15:39:31] E 28/99 | train_loss=0.1413 | val_loss=0.1935 | val_dice=0.8427 | val_iou=0.7282

INFO:pytorch_lightning.callbacks.early_stopping:Metric val_dice improved by 0.000 >= min_delta = 0.0. New best score: 0.843


[15:39:53] E 29/99 | train_loss=0.1400 | val_loss=0.1941 | val_dice=0.8430 | val_iou=0.7285

INFO:pytorch_lightning.callbacks.early_stopping:Metric val_dice improved by 0.000 >= min_delta = 0.0. New best score: 0.843


## Cell 8 — Per-fold history plots

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

for fold, res in results.items():
    if "error" in res:
        print(f"fold {fold}: skipped (failed during training)")
        continue
    h = res.get("history")
    if h is None or h.empty:
        print(f"fold {fold}: no metrics.csv found.")
        continue

    h_epoch = h.dropna(subset=["epoch"]).copy()
    h_epoch["epoch"] = h_epoch["epoch"].astype(int)

    train_curve = h_epoch.dropna(subset=["train_loss"]).groupby("epoch")["train_loss"].last()
    val_loss    = h_epoch.dropna(subset=["val_loss"]).groupby("epoch")["val_loss"].last()
    val_dice    = h_epoch.dropna(subset=["val_dice"]).groupby("epoch")["val_dice"].last()

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(train_curve.index, train_curve.values, label="train_loss")
    axes[0].plot(val_loss.index,    val_loss.values,    label="val_loss")
    axes[0].set_xlabel("epoch"); axes[0].set_ylabel("loss"); axes[0].legend(loc="upper right")
    axes[0].set_title(f"{EXPERIMENT['name']} fold {fold} — losses")
    axes[0].grid(alpha=0.3)

    axes[1].plot(val_dice.index, val_dice.values, label="val_dice", color="green")
    axes[1].axhline(y=val_dice.max(), linestyle="--", alpha=0.4, color="green",
                    label=f"best={val_dice.max():.4f}")
    axes[1].set_xlabel("epoch"); axes[1].set_ylabel("dice"); axes[1].legend(loc="lower right")
    axes[1].set_title(f"{EXPERIMENT['name']} fold {fold} — val_dice")
    axes[1].grid(alpha=0.3)

    fig.tight_layout()

    # Save under outputs/figures/<task>/<dataset>/<exp>/fold_<k>/
    paths = res["paths"]
    fig_dir = paths.get("figures") if "figures" in paths else (
        LOCAL_ROOT / "outputs" / "figures" / EXPERIMENT["task"]
        / EXPERIMENT["dataset"] / EXPERIMENT["name"] / f"fold_{fold}"
    )
    Path(fig_dir).mkdir(parents=True, exist_ok=True)
    out_png = Path(fig_dir) / f"history_fold_{fold}.png"
    fig.savefig(out_png, dpi=120, bbox_inches="tight")
    plt.show()
    print(f"  saved {out_png}")

## Cell 9 — <exp>_train_summary.csv



In [ ]:
import pandas as pd

rows = []
for fold, res in results.items():
    if "error" in res:
        rows.append({
            "experiment":    EXPERIMENT["name"],
            "dataset":       EXPERIMENT["dataset"],
            "split_scheme":  EXPERIMENT["split_scheme"],
            "fold":          fold,
            "status":        "FAILED",
            "error":         res["error"],
        })
        continue
    rows.append({
        "experiment":           EXPERIMENT["name"],
        "dataset":              EXPERIMENT["dataset"],
        "split_scheme":         EXPERIMENT["split_scheme"],
        "fold":                 fold,
        "status":               "OK",
        "best_epoch":           res["best_epoch"],
        "best_val_dice":        res["best_val_dice"],
        "total_epochs_trained": res["total_epochs_trained"],
        "train_minutes":        round(res["train_seconds"] / 60.0, 2),
        "params_count":         res["params_count"],
        "peak_gpu_mem_mb":      res["peak_gpu_mem_mb"],
    })

summary_df = pd.DataFrame(rows)

# Write to outputs/tables/<task>/<dataset>/<exp>/<exp>_train_summary.csv
from pathlib import Path
exp_tables_dir = (
    LOCAL_ROOT / "outputs" / "tables"
    / EXPERIMENT["task"] / EXPERIMENT["dataset"] / EXPERIMENT["name"]
)
exp_tables_dir.mkdir(parents=True, exist_ok=True)
out_csv = exp_tables_dir / f"{EXPERIMENT['name']}_train_summary.csv"
summary_df.to_csv(out_csv, index=False)
print(f"saved {out_csv}")
summary_df

## Cell 10 — Sync to Drive

In [ ]:
from src.notebook_setup import sync_outputs_to_drive

sync_outputs_to_drive(
    drive_root=DRIVE_ROOT, local_root=LOCAL_ROOT,
    task=EXPERIMENT["task"], dataset=EXPERIMENT["dataset"],
    experiment_name=EXPERIMENT["name"],
    categories=["checkpoints", "logs", "tables", "figures"],
)
print("sync complete")

In [ ]:
# Only terminate if sync above actually completed successfully
SYNC_OK = True   # set this manually based on the sync cell's output
if SYNC_OK:
    from google.colab import runtime
    print("Disconnecting runtime in 3 seconds...")
    import time; time.sleep(3)
    runtime.unassign()
else:
    print("SYNC_OK is False — staying connected. Investigate the sync cell first.")# ----- Terminate the Colab session -------------------------------------------


In [ ]:
# ----- Terminate the Colab session -------------------------------------------
# Runs only after sync_outputs_to_drive() above has completed, so everything
# is safely on Drive. After this call:
#   - the runtime disconnects
#   - all in-memory state is lost
#   - compute units stop being consumed
#   - the local SSD copy under /content/Senior_Project_local is wiped
# Reconnecting later starts a fresh runtime (re-run from Cell 1).

from google.colab import runtime
print("All folds complete. Disconnecting runtime in 3 seconds to save compute...")
import time; time.sleep(3)
runtime.unassign()